In [ ]:
import torch
from torch import nn
torch.cuda.is_available()

In [12]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="../runs/dataset")

# Prep

## Datasets

In [ ]:
from lib.datasets import Real, Synthetic, RealXRayClassifier

len(Real("../preprocess/GAN-dataset")), len(Synthetic("../preprocess/GAN-dataset"))

In [ ]:
# 410 because we appended one instance for the frontal xray
# and another for the lateral xray. Therefore there are actually 205
# patients with both view of the xray
len(RealXRayClassifier("../preprocess/GAN-dataset"))

# Classification Real

In [15]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="../runs/real", comment="Real dataset")

In [ ]:
import torch
from lib.datasets import RealXRayClassifier

dataset_class = RealXRayClassifier
dataset = dataset_class("../preprocess/GAN-dataset")
xray, target = dataset[5]
xray.shape, target

In [17]:
from pathlib import Path
SAVE_ROOT = Path("../models/pretrain").resolve()
TASK_FOLDER = SAVE_ROOT / "classify"
SAVE_PATH = TASK_FOLDER / str(dataset_class.__name__)
SAVE_PATH.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image
from torchvision.transforms import v2 as transforms

dataset = dataset_class("../preprocess/GAN-dataset")
xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    #assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 2
    #assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1
    xray = torch.stack([xray, xray, xray],0)
    xray = transforms.Resize((200,200), antialias=True)(xray)
    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("dataset", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/dataset.jpeg", normalize=True)

mean, std, max, min

In [19]:
from torchvision.transforms import v2 as transforms
#mean = torch.tensor(106.3243)
#std = torch.tensor(59.2491)

transform = transforms.Compose([
    transforms.Compose([transforms.ToImageTensor(), transforms.ConvertImageDtype()]),
    transforms.Normalize(mean=[mean, mean, mean], std=[std, std, std]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomChoice([
        transforms.Pad(x) for x in range(20, 50, 20)
    ]),
    transforms.RandomRotation(15),
    transforms.AugMix(),
    transforms.Resize((200,200), antialias=True),
])
dataset = dataset_class("../preprocess/GAN-dataset",transform)

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image

xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 3
    assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1

    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("transforms", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/transforms.jpeg")

mean, std, max, min

In [21]:
from torch.utils.data import DataLoader, random_split

train = 0.8
val = 0.1
test = round(1 - (train + val),4)

In [22]:
generator = torch.Generator().manual_seed(42)

train, val, test = random_split(
    dataset=dataset,
    lengths=[train, val, test],
    generator=generator,
)

In [23]:
datasets = {
    "train": train,
    "val": val,
    "test": test
}

In [24]:
dataloaders = {x:DataLoader(datasets[x],batch_size=64,shuffle=True, num_workers=4) for x in datasets}

# for tqdm
data_lengths = {x: len(datasets[x]) for x in datasets}

## Model

In [ ]:
import timm
classification_model = timm.create_model(
    model_name="convnextv2_atto",
    pretrained=True
)
classification_model.head

In [ ]:
classification_model

In [ ]:
import torch.nn as nn

new_layer = nn.Linear(
    in_features=classification_model.head.fc.in_features,
    out_features=2
)
classification_model.head.fc = new_layer
classification_model.head

In [28]:
writer.add_graph(
    model=classification_model,
    input_to_model=torch.randn(size=(1,3,200,200))
)

In [29]:
model: nn.Module = classification_model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## Optimizers and loss

In [30]:
from torch import nn
from torch import optim

epochs = 15
criterion = nn.CrossEntropyLoss()

In [31]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    betas=(0.9,0.999),
    weight_decay=0.05,
)

In [32]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=epochs
)

## Training

In [33]:
from torch.utils.data import DataLoader
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall, binary_auprc, binary_auroc
from tqdm import tqdm
from lib.utils.utils import DataStorer
import warnings

storer = DataStorer()

def train_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler.LRScheduler,
    epoch: int,
):
    phase = "train"
    model.train()

    pbar = tqdm(dataloaders[phase],f"train - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_loss = 0
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)

            optimizer.zero_grad()
            
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            running_loss += loss
            loss.backward()

            optimizer.step()
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        storer.store("lr", optimizer.param_groups[0]["lr"])
        scheduler.step()
        avg_loss = running_loss/len(dataloaders[phase])
        writer.add_scalar("train_loss",avg_loss, epoch)
        storer.store("train_loss", avg_loss)
    pbar.close()
    return avg_loss


def val_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,  
    epoch: int,     
):
    phase = "val"
    model.eval()
    pbar = tqdm(dataloaders[phase],f"val - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_metrics = {}
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)
            
            writer.add_text("val_target",str(targets.t()), epoch)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            writer.add_text("val_outputs",str(predicted), epoch)
            
            loss = criterion(outputs, targets)

            precision = binary_precision(predicted, targets)
            recall = binary_recall(predicted, targets)
            accuracy = binary_accuracy(predicted, targets)
            f1 = binary_f1_score(predicted, targets)
            auroc = binary_auroc(predicted, targets)
            auprc = binary_auprc(predicted, targets)
            
            metrics = [
                precision,
                recall,
                accuracy,
                f1,
                auroc,
                auprc,
                loss,
            ]
            metric_names = [
                "precision",
                "recall",
                "accuracy",
                "f1",
                "auroc",
                "auprc",
                "val_loss",
            ]

            for metric_name, metric in zip(metric_names, metrics):
                if metric_name not in running_metrics:
                    running_metrics[metric_name] = metric.clone()
                    continue
                running_metrics[metric_name] += metric

        for metric_name in metric_names:
            avg = running_metrics[metric_name] / len(dataloaders[phase])
            writer.add_scalar(metric_name,avg,epoch)
            storer.store(metric_name, avg)
        f1 = running_metrics["f1"] / len(dataloaders[phase])
    pbar.close()
    return f1

In [ ]:
pbar = tqdm(range(epochs))
best = None
for i in pbar:
    pbar.set_description(f"Epoch {i}/{epochs}")
    train_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=i
    )
    f1 = val_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        epoch=i
    )
    if best is None:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "f1": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    if f1 > best:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    
    torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best,
            },
            SAVE_PATH / "latest.pt"
        )


In [ ]:
import matplotlib.pyplot as plt

for metric in storer.data:
    for idx, data_point in enumerate(storer.data[metric].data):
        if not isinstance(data_point, torch.Tensor):
            continue
        storer.data[metric].data[idx] = data_point.to("cpu").detach()

storer.write_csv(SAVE_PATH / "metrics.tsv")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="loss",
    title="Train/Validation Losses",
    tags=["train_loss", "val_loss"]
)
losses.savefig(SAVE_PATH / "losses.svg")
losses.savefig(SAVE_PATH / "losses.png")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "val_metrics.svg")
losses.savefig(SAVE_PATH / "val_metrics.png")

In [52]:
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall
from torchmetrics.classification import BinaryROC
ckpt = torch.load(SAVE_PATH / "best.pt")
model.load_state_dict(ckpt["weights"])

def show(imgs):
    if not isinstance(imgs, list):
        imgs = [imgs]
    fig, axs = plt.subplots(ncols=len(imgs), squeeze=False)
    for i, img in enumerate(imgs):
        img = img.detach()
        img = F.to_pil_image(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

storer = DataStorer()
running_metrics = {}
model.eval()
for batch in dataloaders["val"]:
    inputs = batch[0].to(device)
    target = batch[1].to(device)

    outputs = model(inputs)
    if len(torch.unique(outputs)) == 1:
        warnings.warn("Model is not converging correctly")
    _, predicted = torch.max(outputs, 1)
    preds = torch.nn.functional.sigmoid(outputs[:, 1])
    

    precision = binary_precision(predicted, target)
    recall = binary_recall(predicted, target)
    accuracy = binary_accuracy(predicted, target)
    f1 = binary_f1_score(predicted, target)
    auroc = binary_auroc(predicted, target)
    auprc = binary_auprc(predicted, target)

    metrics = [
        precision,
        recall,
        accuracy,
        f1,
        auroc,
        auprc,
    ]
    metric_names = [
        "precision",
        "recall",
        "accuracy",
        "f1",
        "auroc",
        "auprc",
    ]

    for metric_name, metric in zip(metric_names, metrics):
        if metric_name not in running_metrics:
            running_metrics[metric_name] = metric.clone()
            continue
        running_metrics[metric_name] += metric

    grid = make_grid(inputs)
    writer.add_image("test", grid)

    #roc = BinaryROC()
    #roc.update(preds,target)
    #fig_, ax_ = roc.plot(score=True)
    #plt.show()
    #show(grid)
for metric_name in metric_names:
    avg = running_metrics[metric_name] / len(dataloaders["test"])
    writer.add_scalar(metric_name,avg)
    storer.store(metric_name, avg)

In [ ]:
losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics Over Epochs",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "test_metrics.svg")
losses.savefig(SAVE_PATH / "test_metrics.png")

# Classification Synthetic

In [54]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="../runs/synthetic", comment="Synthetic dataset")

In [ ]:
import torch
from lib.datasets import SyntheticXRayClassifier

dataset_class = SyntheticXRayClassifier
dataset = dataset_class("../preprocess/GAN-dataset")
xray, target = dataset[5]
xray.shape, target

In [56]:
from pathlib import Path
SAVE_ROOT = Path("../models/pretrain").resolve()
TASK_FOLDER = SAVE_ROOT / "classify"
SAVE_PATH = TASK_FOLDER / str(dataset_class.__name__)
SAVE_PATH.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image
from torchvision.transforms import v2 as transforms

dataset = dataset_class("../preprocess/GAN-dataset")
xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    #assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 2
    #assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1
    xray = torch.stack([xray, xray, xray],0)
    xray = transforms.Resize((200,200), antialias=True)(xray)
    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("dataset", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/dataset.jpeg", normalize=True)

mean, std, max, min

In [58]:
from torchvision.transforms import v2 as transforms
#mean = torch.tensor(106.3243)
#std = torch.tensor(59.2491)

transform = transforms.Compose([
    transforms.Compose([transforms.ToImageTensor(), transforms.ConvertImageDtype()]),
    transforms.Normalize(mean=[mean, mean, mean], std=[std, std, std]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomChoice([
        transforms.Pad(x) for x in range(20, 50, 20)
    ]),
    transforms.RandomRotation(15),
    transforms.AugMix(),
    transforms.Resize((200,200), antialias=True),
])
dataset = dataset_class("../preprocess/GAN-dataset",transform)

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image

xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 3
    assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1

    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("transforms", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/transforms.jpeg")

mean, std, max, min

In [60]:
from torch.utils.data import DataLoader, random_split

train = 0.8
val = 0.1
test = round(1 - (train + val),4)

In [61]:
generator = torch.Generator().manual_seed(42)

train, val, test = random_split(
    dataset=dataset,
    lengths=[train, val, test],
    generator=generator,
)

In [62]:
datasets = {
    "train": train,
    "val": val,
    "test": test
}

In [63]:
dataloaders = {x:DataLoader(datasets[x],batch_size=64,shuffle=True, num_workers=4) for x in datasets}

# for tqdm
data_lengths = {x: len(datasets[x]) for x in datasets}

## Model

In [ ]:
import timm
classification_model = timm.create_model(
    model_name="convnextv2_atto",
    pretrained=True
)
classification_model.head

In [ ]:
classification_model

In [ ]:
import torch.nn as nn

new_layer = nn.Linear(
    in_features=classification_model.head.fc.in_features,
    out_features=2
)
classification_model.head.fc = new_layer
classification_model.head

In [67]:
writer.add_graph(
    model=classification_model,
    input_to_model=torch.randn(size=(1,3,200,200))
)

In [68]:
model: nn.Module = classification_model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## Optimizers and loss

In [69]:
from torch import nn
from torch import optim

epochs = 15
criterion = nn.CrossEntropyLoss()

In [70]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    betas=(0.9,0.999),
    weight_decay=0.05,
)

In [71]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=epochs
)

## Training

In [72]:
from torch.utils.data import DataLoader
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall, binary_auprc, binary_auroc
from tqdm import tqdm
from lib.utils.utils import DataStorer
import warnings

storer = DataStorer()

def train_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler.LRScheduler,
    epoch: int,
):
    phase = "train"
    model.train()

    pbar = tqdm(dataloaders[phase],f"train - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_loss = 0
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)

            optimizer.zero_grad()
            
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            running_loss += loss
            loss.backward()

            optimizer.step()
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        storer.store("lr", optimizer.param_groups[0]["lr"])
        scheduler.step()
        avg_loss = running_loss/len(dataloaders[phase])
        writer.add_scalar("train_loss",avg_loss, epoch)
        storer.store("train_loss", avg_loss)
    pbar.close()
    return avg_loss


def val_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,  
    epoch: int,     
):
    phase = "val"
    model.eval()
    pbar = tqdm(dataloaders[phase],f"val - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_metrics = {}
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)
            
            writer.add_text("val_target",str(targets.t()), epoch)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            writer.add_text("val_outputs",str(predicted), epoch)
            
            loss = criterion(outputs, targets)

            precision = binary_precision(predicted, targets)
            recall = binary_recall(predicted, targets)
            accuracy = binary_accuracy(predicted, targets)
            f1 = binary_f1_score(predicted, targets)
            auroc = binary_auroc(predicted, targets)
            auprc = binary_auprc(predicted, targets)
            
            metrics = [
                precision,
                recall,
                accuracy,
                f1,
                auroc,
                auprc,
                loss,
            ]
            metric_names = [
                "precision",
                "recall",
                "accuracy",
                "f1",
                "auroc",
                "auprc",
                "val_loss",
            ]

            for metric_name, metric in zip(metric_names, metrics):
                if metric_name not in running_metrics:
                    running_metrics[metric_name] = metric.clone()
                    continue
                running_metrics[metric_name] += metric

        for metric_name in metric_names:
            avg = running_metrics[metric_name] / len(dataloaders[phase])
            writer.add_scalar(metric_name,avg,epoch)
            storer.store(metric_name, avg)
        f1 = running_metrics["f1"] / len(dataloaders[phase])
    pbar.close()
    return f1

In [ ]:
pbar = tqdm(range(epochs))
best = None
for i in pbar:
    pbar.set_description(f"Epoch {i}/{epochs}")
    train_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=i
    )
    f1 = val_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        epoch=i
    )
    if best is None:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "f1": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    if f1 > best:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    
    torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best,
            },
            SAVE_PATH / "latest.pt"
        )


In [ ]:
import matplotlib.pyplot as plt

for metric in storer.data:
    for idx, data_point in enumerate(storer.data[metric].data):
        if not isinstance(data_point, torch.Tensor):
            continue
        storer.data[metric].data[idx] = data_point.to("cpu").detach()

storer.write_csv(SAVE_PATH / "metrics.tsv")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="loss",
    title="Train/Validation Losses",
    tags=["train_loss", "val_loss"]
)
losses.savefig(SAVE_PATH / "losses.svg")
losses.savefig(SAVE_PATH / "losses.png")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "val_metrics.svg")
losses.savefig(SAVE_PATH / "val_metrics.png")

In [ ]:
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall
from torchmetrics.classification import BinaryROC
ckpt = torch.load(SAVE_PATH / "best.pt")
model.load_state_dict(ckpt["weights"])

def show(imgs):
    if not isinstance(imgs, list):
        imgs = [imgs]
    fig, axs = plt.subplots(ncols=len(imgs), squeeze=False)
    for i, img in enumerate(imgs):
        img = img.detach()
        img = F.to_pil_image(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

storer = DataStorer()
running_metrics = {}
model.eval()
for batch in dataloaders["val"]:
    inputs = batch[0].to(device)
    target = batch[1].to(device)

    outputs = model(inputs)
    if len(torch.unique(outputs)) == 1:
        warnings.warn("Model is not converging correctly")
    _, predicted = torch.max(outputs, 1)
    preds = torch.nn.functional.sigmoid(outputs[:, 1])
    

    precision = binary_precision(predicted, target)
    recall = binary_recall(predicted, target)
    accuracy = binary_accuracy(predicted, target)
    f1 = binary_f1_score(predicted, target)
    auroc = binary_auroc(predicted, target)
    auprc = binary_auprc(predicted, target)

    metrics = [
        precision,
        recall,
        accuracy,
        f1,
        auroc,
        auprc,
    ]
    metric_names = [
        "precision",
        "recall",
        "accuracy",
        "f1",
        "auroc",
        "auprc",
    ]

    for metric_name, metric in zip(metric_names, metrics):
        if metric_name not in running_metrics:
            running_metrics[metric_name] = metric.clone()
            continue
        running_metrics[metric_name] += metric

    grid = make_grid(inputs)
    writer.add_image("test", grid)

    #roc = BinaryROC()
    #roc.update(preds,target)
    #fig_, ax_ = roc.plot(score=True)
    #plt.show()
    #show(grid)
for metric_name in metric_names:
    avg = running_metrics[metric_name] / len(dataloaders["test"])
    writer.add_scalar(metric_name,avg)
    storer.store(metric_name, avg)

In [ ]:
losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics Over Epochs",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "test_metrics.svg")
losses.savefig(SAVE_PATH / "test_metrics.png")

# Classification Mixed

In [77]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="../runs/mixed", comment="Mixed dataset")

In [ ]:
import torch
from lib.datasets import MixedXRayClassifier

dataset_class = MixedXRayClassifier
dataset = dataset_class("../preprocess/GAN-dataset")
xray, target = dataset[5]
xray.shape, target

In [79]:
from pathlib import Path
SAVE_ROOT = Path("../models/pretrain").resolve()
TASK_FOLDER = SAVE_ROOT / "classify"
SAVE_PATH = TASK_FOLDER / str(dataset_class.__name__)
SAVE_PATH.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image
from torchvision.transforms import v2 as transforms

dataset = dataset_class("../preprocess/GAN-dataset")
xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    #assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 2
    #assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1
    xray = torch.stack([xray, xray, xray],0)
    xray = transforms.Resize((200,200), antialias=True)(xray)
    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("dataset", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/dataset.jpeg", normalize=True)

mean, std, max, min

In [81]:
from torchvision.transforms import v2 as transforms
#mean = torch.tensor(106.3243)
#std = torch.tensor(59.2491)

transform = transforms.Compose([
    transforms.Compose([transforms.ToImageTensor(), transforms.ConvertImageDtype()]),
    transforms.Normalize(mean=[mean, mean, mean], std=[std, std, std]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomChoice([
        transforms.Pad(x) for x in range(20, 50, 20)
    ]),
    transforms.RandomRotation(15),
    transforms.AugMix(),
    transforms.Resize((200,200), antialias=True),
])
dataset = dataset_class("../preprocess/GAN-dataset",transform)

In [ ]:
from tqdm import tqdm
from torchvision.utils import make_grid, save_image

xrays = []
for xray, target in tqdm(dataset):

    # xray tests
    assert xray.dtype == torch.float
    # shape should be (3, 200, 200)
    assert xray.shape[1] == xray.shape[2] == 200
    assert len(xray.shape) == 3
    assert xray.shape[0] == 3

    # target tests
    #assert target.dtype == torch.float
    # shape should be (1)
    #assert len(target.shape) == 1
    #assert target.shape[0] == 1

    xrays.append(xray)

grid = make_grid(
    tensor=xrays
)

mean = grid.mean()
std = grid.std()
max = grid.max()
min = grid.min()
writer.add_image("transforms", grid,dataformats="CHW")
save_image(grid,"../output/models/pretrain/classify/transforms.jpeg")

mean, std, max, min

In [83]:
from torch.utils.data import DataLoader, random_split

train = 0.8
val = 0.1
test = round(1 - (train + val),4)

In [84]:
generator = torch.Generator().manual_seed(42)

train, val, test = random_split(
    dataset=dataset,
    lengths=[train, val, test],
    generator=generator,
)

In [85]:
datasets = {
    "train": train,
    "val": val,
    "test": test
}

In [86]:
dataloaders = {x:DataLoader(datasets[x],batch_size=64,shuffle=True) for x in datasets}

# for tqdm
data_lengths = {x: len(datasets[x]) for x in datasets}

## Model

In [ ]:
import timm
classification_model = timm.create_model(
    model_name="convnextv2_atto",
    pretrained=True
)
classification_model.head

In [ ]:
classification_model

In [ ]:
import torch.nn as nn

new_layer = nn.Linear(
    in_features=classification_model.head.fc.in_features,
    out_features=2
)
classification_model.head.fc = new_layer
classification_model.head

In [90]:
writer.add_graph(
    model=classification_model,
    input_to_model=torch.randn(size=(1,3,200,200))
)

In [91]:
model: nn.Module = classification_model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## Optimizers and loss

In [92]:
from torch import nn
from torch import optim

epochs = 20
criterion = nn.CrossEntropyLoss()

In [93]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    betas=(0.9,0.999),
    weight_decay=0.05,
)

In [94]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=epochs
)

## Training

In [95]:
from torch.utils.data import DataLoader
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall, binary_auprc, binary_auroc
from tqdm import tqdm
from lib.utils.utils import DataStorer
import warnings

storer = DataStorer()

def train_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler.LRScheduler,
    epoch: int,
):
    phase = "train"
    model.train()

    pbar = tqdm(dataloaders[phase],f"train - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_loss = 0
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)

            optimizer.zero_grad()
            
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            running_loss += loss
            loss.backward()

            optimizer.step()
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        storer.store("lr", optimizer.param_groups[0]["lr"])
        scheduler.step()
        avg_loss = running_loss/len(dataloaders[phase])
        writer.add_scalar("train_loss",avg_loss, epoch)
        storer.store("train_loss", avg_loss)
    pbar.close()
    return avg_loss


def val_one_epoch(
    model: nn.Module,
    dataloaders: dict[str, DataLoader],
    criterion: nn.Module,  
    epoch: int,     
):
    phase = "val"
    model.eval()
    pbar = tqdm(dataloaders[phase],f"val - epoch {epoch}")
    
    with torch.set_grad_enabled(phase == "train"):
        running_metrics = {}
        for batch in pbar:
            inputs: torch.Tensor = batch[0].to(device)
            targets: torch.Tensor = batch[1].to(device)
            
            writer.add_text("val_target",str(targets.t()), epoch)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            writer.add_text("val_outputs",str(predicted), epoch)
            
            loss = criterion(outputs, targets)

            precision = binary_precision(predicted, targets)
            recall = binary_recall(predicted, targets)
            accuracy = binary_accuracy(predicted, targets)
            f1 = binary_f1_score(predicted, targets)
            auroc = binary_auroc(predicted, targets)
            auprc = binary_auprc(predicted, targets)
            
            metrics = [
                precision,
                recall,
                accuracy,
                f1,
                auroc,
                auprc,
                loss,
            ]
            metric_names = [
                "precision",
                "recall",
                "accuracy",
                "f1",
                "auroc",
                "auprc",
                "val_loss",
            ]

            for metric_name, metric in zip(metric_names, metrics):
                if metric_name not in running_metrics:
                    running_metrics[metric_name] = metric.clone()
                    continue
                running_metrics[metric_name] += metric

        for metric_name in metric_names:
            avg = running_metrics[metric_name] / len(dataloaders[phase])
            writer.add_scalar(metric_name,avg,epoch)
            storer.store(metric_name, avg)
        f1 = running_metrics["f1"] / len(dataloaders[phase])
    pbar.close()
    return f1

In [ ]:
pbar = tqdm(range(epochs))
best = None
for i in pbar:
    pbar.set_description(f"Epoch {i}/{epochs}")
    train_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=i
    )
    f1 = val_one_epoch(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        epoch=i
    )
    if best is None:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "f1": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    if f1 > best:
        best = f1
        tqdm.write(f"New best: {best}")
        torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best
            },
            SAVE_PATH / "best.pt"
        )
    
    torch.save(
            {
                "epoch": i,
                "val_loss": f1,
                "weights": model.state_dict(),
                "best": best,
            },
            SAVE_PATH / "latest.pt"
        )


In [ ]:
import matplotlib.pyplot as plt

for metric in storer.data:
    for idx, data_point in enumerate(storer.data[metric].data):
        if not isinstance(data_point, torch.Tensor):
            continue
        storer.data[metric].data[idx] = data_point.to("cpu").detach()

storer.write_csv(SAVE_PATH / "metrics.tsv")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="loss",
    title="Train/Validation Losses",
    tags=["train_loss", "val_loss"]
)
losses.savefig(SAVE_PATH / "losses.svg")
losses.savefig(SAVE_PATH / "losses.png")

losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "val_metrics.svg")
losses.savefig(SAVE_PATH / "val_metrics.png")

In [98]:
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from torcheval.metrics.functional import binary_accuracy, binary_f1_score, binary_precision, binary_recall
from torchmetrics.classification import BinaryROC
ckpt = torch.load(SAVE_PATH / "best.pt")
model.load_state_dict(ckpt["weights"])

def show(imgs):
    if not isinstance(imgs, list):
        imgs = [imgs]
    fig, axs = plt.subplots(ncols=len(imgs), squeeze=False)
    for i, img in enumerate(imgs):
        img = img.detach()
        img = F.to_pil_image(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

storer = DataStorer()
running_metrics = {}
model.eval()
for batch in dataloaders["val"]:
    inputs = batch[0].to(device)
    target = batch[1].to(device)

    outputs = model(inputs)
    if len(torch.unique(outputs)) == 1:
        warnings.warn("Model is not converging correctly")
    _, predicted = torch.max(outputs, 1)
    preds = torch.nn.functional.sigmoid(outputs[:, 1])
    

    precision = binary_precision(predicted, target)
    recall = binary_recall(predicted, target)
    accuracy = binary_accuracy(predicted, target)
    f1 = binary_f1_score(predicted, target)
    auroc = binary_auroc(predicted, target)
    auprc = binary_auprc(predicted, target)

    metrics = [
        precision,
        recall,
        accuracy,
        f1,
        auroc,
        auprc,
    ]
    metric_names = [
        "precision",
        "recall",
        "accuracy",
        "f1",
        "auroc",
        "auprc",
    ]

    for metric_name, metric in zip(metric_names, metrics):
        if metric_name not in running_metrics:
            running_metrics[metric_name] = metric.clone()
            continue
        running_metrics[metric_name] += metric

    grid = make_grid(inputs)
    writer.add_image("test", grid)

    #roc = BinaryROC()
    #roc.update(preds,target)
    #fig_, ax_ = roc.plot(score=True)
    #plt.show()
    #show(grid)
for metric_name in metric_names:
    avg = running_metrics[metric_name] / len(dataloaders["test"])
    writer.add_scalar(metric_name,avg)
    storer.store(metric_name, avg)

In [ ]:
losses: plt = storer.plot(
    xlabel="epoch",
    ylabel="metric",
    title="Validation Metrics Over Epochs",
    tags=["f1", "auroc", "auprc"]
)
losses.savefig(SAVE_PATH / "test_metrics.svg")
losses.savefig(SAVE_PATH / "test_metrics.png")